# mapping-sim: features, matches and tracks against the truth

The first layer of an OpenSfM run on a `mapping_sim` render -- detection, pairwise matching,
track building -- scored by construction: every feature's ray is cast through the camera that
rendered it, so each one has the surface point, solid and range it truly saw. A match is then
right or wrong by geometry and by entity, not by whether the reconstruction later converged.
Formats and conventions: `mapping_sim/opensfm_data.py`. Experiments: `mapping-sim-experiments.md`.

In [ ]:
run_dir = "/home/jovyan/datasets/mapping-sim/runs/map-test-20261105-1551"
wrong_px = 10.0  # a match whose true reprojection error exceeds this (full-resolution px) counts as wrong
impure_m = 0.25  # a track whose members' true points sit further than this from their mean counts as impure
debug = False

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import io, json, pathlib, sys, time, collections
from IPython.display import display as ipy_display, Image as ipy_image

sys.path.insert(0, 'analysis/mapping_sim')  # run from the repo root, as the flight-analysis runner does
import opensfm_data as od
from rig import rig_from_config
from scene import Scene

plt.rcParams.update({'figure.dpi': 110, 'font.size': 9})

def show(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight')
    plt.close(fig)
    ipy_display(ipy_image(data=buf.getvalue()))

RUN = pathlib.Path(run_dir)
OSFM = RUN / 'opensfm'
run = json.loads((RUN / 'run.json').read_text())
RENDER = RUN.parent.parent / 'renders' / run['render']
truth = json.loads((RENDER / 'truth.json').read_text())
cfg = truth['config']
cams = {c.name: c for c in rig_from_config(cfg['rig'])}
if truth['scale'] != 1.0:
    for c in cams.values():
        c.lens = c.lens.scaled(truth['scale'])
scene = Scene.from_config(cfg['scene'])
# What each feature saw, as a code into LABELS: per solid, then 'house star' and 'sky' (solid -1).
solid_names = od.solid_labels(cfg['scene'])
LABELS = ['ground', 'house', 'house star'] + sorted({l for l in solid_names if l.startswith('tree')}, key=lambda s: float(s.split('=')[1]))
LABELS += sorted(set(solid_names) - set(LABELS)) + ['sky']
solid_code = np.array([LABELS.index(l) for l in solid_names] + [LABELS.index('sky')], dtype=np.int8)
STAR = LABELS.index('house star')
def named(codes, name=None):
    return pd.Series(pd.Categorical.from_codes(codes, LABELS), name=name)
names = sorted(truth['shots'])
index = {n: i for i, n in enumerate(names)}
shot = [truth['shots'][n] for n in names]
cam_of = np.array([s['camera'] for s in shot])
capture = np.array([int(n.split('_')[0]) for n in names])
print(f"run {RUN.name}: render {run['render']}, {len(names)} images, cameras {sorted(cams)}, {len(scene.solids)} solids")
print(f"ODM {run['odm_image']} args {run['odm_args']!r}")
print((OSFM / 'profile.log').read_text())

## Assumptions

In [ ]:
# Every rendered image has features and matches, and the tracks file is the version the reader knows.
missing = [n for n in names if not (OSFM / 'features' / f'{n}.features.npz').exists()]
assert not missing, f"no features for {len(missing)} images, e.g. {missing[:3]}"
missing = [n for n in names if not (OSFM / 'matches' / f'{n}_matches.pkl.gz').exists()]
assert not missing, f"no matches for {len(missing)} images, e.g. {missing[:3]}"
assert (OSFM / 'tracks.csv').read_bytes()[:len(od.TRACKS_HEADER)] == od.TRACKS_HEADER

# The pixel convention: a feature's stored colour is the rendered pixel at to_pixels(). If this
# fails, every truth below is cast through the wrong pixel.
from PIL import Image
rng = np.random.default_rng(0)
for n in rng.choice(names, 6, replace=False):
    lens = cams[truth['shots'][n]['camera']].lens
    with np.load(OSFM / 'features' / f'{n}.features.npz') as f:
        uv = od.to_pixels(f['points'], lens.width, lens.height)
        col = f['colors'][:, 0].astype(float)
    img = np.asarray(Image.open(RENDER / 'images' / n).convert('L')).astype(float)
    ij = np.clip(np.floor(uv).astype(int), 0, [lens.width - 1, lens.height - 1])
    r = np.corrcoef(col, img[ij[:, 1], ij[:, 0]])[0, 1]
    print(f"{n}: stored colour vs rendered pixel r = {r:.4f}")
    assert r > 0.98, n

## Detection: what each image's features landed on

In [ ]:
t0 = time.time()
feat_img, feat_uv, feat_scale = [], [], []
truth_cols = collections.defaultdict(list)
offset = np.zeros(len(names) + 1, dtype=np.int64)
for i, n in enumerate(names):
    lens = cams[cam_of[i]].lens
    p = od.read_features(OSFM, n)
    uv = od.to_pixels(p, lens.width, lens.height)
    ft = od.feature_truth(scene, lens, np.array(shot[i]['R_wc']), np.array(shot[i]['center']), uv)
    feat_img.append(np.full(len(p), i, dtype=np.int32))
    feat_uv.append(uv.astype(np.float32))
    feat_scale.append((p[:, 2] * max(lens.width, lens.height)).astype(np.float32))  # px
    for k, v in ft.items():
        truth_cols[k].append(v.astype(np.float32) if v.dtype == np.float64 else v)
    offset[i + 1] = offset[i] + len(p)
F = {'image': np.concatenate(feat_img), 'uv': np.concatenate(feat_uv), 'scale_px': np.concatenate(feat_scale)}
F |= {k: np.concatenate(v) for k, v in truth_cols.items()}
F['lab'] = np.where(F['star'], STAR, solid_code[F['solid']]).astype(np.int8)
del feat_img, feat_uv, feat_scale, truth_cols
per_image = np.diff(offset)
print(f"{len(F['image']):,} features in {time.time() - t0:.0f} s; per image min {per_image.min()} median {np.median(per_image):.0f} max {per_image.max()}")
by_label = named(F['lab']).value_counts(sort=False)
by_label = by_label[by_label > 0]
print((by_label / by_label.sum()).round(4).to_string())

In [ ]:
order = list(by_label.index)
fig, axes = plt.subplots(len(cams), 1, figsize=(12, 2.2 * len(cams)), sharex=True)
fig.suptitle('Features per image by what they landed on', fontweight='bold')
tab = pd.crosstab(F['image'], named(F['lab'])).reindex(columns=order, fill_value=0)
cmap = plt.get_cmap('tab20')
for ax, c in zip(axes, sorted(cams)):
    rows = np.flatnonzero(cam_of == c)
    t = tab.loc[rows]
    ax.stackplot(capture[rows], t.T.values, labels=order, colors=[cmap(k) for k in range(len(order))])
    ax.set_ylabel(c)
    ax.grid(alpha=0.3)
axes[0].legend(fontsize=7, ncol=6, loc='upper right')
axes[-1].set_xlabel('capture')
show(fig)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for l in order:
    m = F['lab'] == LABELS.index(l)
    if l != 'sky':
        axes[0].hist(F['range'][m], bins=np.linspace(0, 150, 76), histtype='step', label=l)
axes[0].set_xlabel('range (m)'); axes[0].set_ylabel('features'); axes[0].legend(fontsize=7); axes[0].grid(alpha=0.3)
axes[1].hist(F['scale_px'], bins=100, log=True)
axes[1].set_xlabel('feature scale (full-res px)'); axes[1].grid(alpha=0.3)
show(fig)

## Matching: every pair OpenSfM tried, and whether each match is the same surface point

In [ ]:
t0 = time.time()
pair_rows = []
M = collections.defaultdict(list)
for i, n in enumerate(names):
    for other, pairs in od.read_matches(OSFM, n).items():
        j = index[other]
        pair_rows.append((i, j, len(pairs)))
        if len(pairs) == 0:
            continue
        a = offset[i] + pairs[:, 0].astype(np.int64)
        b = offset[j] + pairs[:, 1].astype(np.int64)
        lens_b = cams[cam_of[j]].lens
        proj, front = od.project(lens_b, np.array(shot[j]['R_wc']), np.array(shot[j]['center']), F['point'][a].astype(np.float64))
        err = np.linalg.norm(proj - F['uv'][b], axis=1)
        err[~front | np.isnan(err)] = np.inf  # behind B or A saw sky: nothing B could match
        M['a'].append(a); M['b'].append(b); M['err_px'].append(err.astype(np.float32))
pairs = pd.DataFrame(pair_rows, columns=['i', 'j', 'n'])
pairs['gap'] = np.abs(capture[pairs.i] - capture[pairs.j])
pairs['cams'] = [f'{min(x, y)}-{max(x, y)}' for x, y in zip(cam_of[pairs.i], cam_of[pairs.j])]
M = {k: np.concatenate(v) for k, v in M.items()}
M['sep_m'] = np.linalg.norm(F['point'][M['a']] - F['point'][M['b']], axis=1)
M['same_solid'] = F['solid'][M['a']] == F['solid'][M['b']]
M['wrong'] = (M['err_px'] > wrong_px) | ~M['same_solid']
dup = pairs.assign(k=[tuple(sorted(p)) for p in zip(pairs.i, pairs.j)]).k.duplicated().sum()
print(f"{len(pairs):,} pairs tried ({dup} listed from both sides), {int((pairs.n > 0).sum()):,} kept, {len(M['a']):,} matches in {time.time() - t0:.0f} s")
print(f"wrong (true reprojection > {wrong_px:g} px or a different solid): {M['wrong'].sum():,} = {M['wrong'].mean():.3%}")
print(f"true reprojection error, px: p50 {np.median(M['err_px']):.2f}  p90 {np.percentile(M['err_px'], 90):.2f}  p99 {np.percentile(M['err_px'], 99):.2f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
e = np.clip(M['err_px'], 1e-3, 1e4)
axes[0].hist(e, bins=np.logspace(-2, 4, 121), log=True)
axes[0].axvline(wrong_px, color='crimson', ls='--', label=f'wrong_px {wrong_px:g}')
axes[0].set_xscale('log'); axes[0].set_xlabel('true reprojection error in B (full-res px)'); axes[0].legend(); axes[0].grid(alpha=0.3)

g = pairs.groupby('gap').agg(tried=('n', 'size'), kept=('n', lambda x: (x > 0).sum()), matches=('n', 'sum'))
axes[1].bar(g.index, g.tried, color='lightgray', label='tried')
axes[1].bar(g.index, g.kept, color='steelblue', label='kept')
axes[1].set_xlabel('capture gap'); axes[1].set_ylabel('pairs'); axes[1].set_xlim(-1, min(g.index.max(), 300) + 1); axes[1].legend(); axes[1].grid(alpha=0.3)

mg = pd.DataFrame({'gap': np.abs(capture[F['image'][M['a']]] - capture[F['image'][M['b']]]), 'wrong': M['wrong']})
w = mg.groupby('gap').wrong.agg(['mean', 'size'])
axes[2].scatter(w.index, w['mean'], s=np.clip(w['size'] / w['size'].max() * 60, 2, 60))
axes[2].set_xlabel('capture gap'); axes[2].set_ylabel('fraction wrong'); axes[2].set_xlim(-1, min(w.index.max(), 300) + 1); axes[2].grid(alpha=0.3)
show(fig)

print('matches and fraction wrong, by what A saw:')
print(pd.DataFrame({'label': named(F['lab'][M['a']]), 'wrong': M['wrong']}).groupby('label', observed=True).wrong.agg(matches='size', wrong='mean').round(4).to_string())
print()
print('by camera pair:')
print(pairs[pairs.n > 0].groupby('cams').n.agg(pairs='size', matches='sum', median='median').to_string())

In [ ]:
# Wrong matches by what each side truly saw: the off-diagonal is two unrelated surfaces matched.
w = M['wrong']
conf = pd.crosstab(named(F['lab'][M['a'][w]], 'A saw'), named(F['lab'][M['b'][w]], 'B saw'))
print(conf.to_string())

## Tracks: chain length, purity, and how far each one reaches in time

In [ ]:
t0 = time.time()
track_names, T = od.read_tracks(OSFM / 'tracks.csv')
remap = np.array([index[n] for n in track_names])
T['image'] = remap[T['image']]
T['g'] = offset[T['image']] + T['feature']
srt = np.argsort(T['track'], kind='stable')
T = {k: v[srt] for k, v in T.items()}
starts = np.flatnonzero(np.r_[True, T['track'][1:] != T['track'][:-1]])
length = np.diff(np.r_[starts, len(T['track'])])
cap = capture[T['image']]
span = np.maximum.reduceat(cap, starts) - np.minimum.reduceat(cap, starts)
pts = F['point'][T['g']].astype(np.float64)
sky = np.isnan(pts[:, 0])
mean = np.add.reduceat(np.where(sky[:, None], 0, pts), starts) / np.maximum(np.add.reduceat((~sky).astype(int), starts), 1)[:, None]
dist = np.linalg.norm(pts - np.repeat(mean, length, axis=0), axis=1)
spread = np.fmax.reduceat(np.where(sky, np.nan, dist), starts)
sol = F['solid'][T['g']]
solids_per_track = np.array([len(set(x)) for x in np.split(sol, starts[1:])])
caps_per_track = np.array([len(set(x)) for x in np.split(cap, starts[1:])])
tracks = pd.DataFrame({'length': length, 'captures': caps_per_track, 'span': span, 'spread_m': spread, 'solids': solids_per_track,
                       'label': named(F['lab'][T['g'][starts]])})
tracks['impure'] = (tracks.solids > 1) | (tracks.spread_m > impure_m)
print(f"{len(tracks):,} tracks, {len(T['track']):,} observations, read in {time.time() - t0:.0f} s")
print(tracks[['length', 'captures', 'span', 'spread_m']].describe(percentiles=[.1, .5, .9, .99]).round(3).to_string())
print(f"tracks touching more than one solid: {(tracks.solids > 1).mean():.3%};  spread over {impure_m:g} m: {(tracks.spread_m > impure_m).mean():.3%}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
L = np.bincount(tracks.length)
axes[0].bar(np.arange(len(L)), L, log=True); axes[0].set_xlabel('track length (observations)'); axes[0].set_xlim(0, min(len(L), 80)); axes[0].grid(alpha=0.3)
S = np.bincount(tracks.span)
axes[1].bar(np.arange(len(S)), S, log=True); axes[1].set_xlabel('capture span (last - first)'); axes[1].grid(alpha=0.3)
im = tracks.groupby(np.minimum(tracks.length, 40)).impure.mean()
axes[2].plot(im.index, im.values, 'o-'); axes[2].set_xlabel('track length (40 = 40+)'); axes[2].set_ylabel('fraction impure'); axes[2].grid(alpha=0.3)
show(fig)

## Starvation: how much of each image's feature budget ends up in a track, and where

In [ ]:
in_track = np.zeros(len(F['image']), dtype=bool)
in_track[T['g']] = True
# Per capture, features in tracks that reach another capture, and one at least 5 captures away.
tr_of_obs = np.repeat(np.arange(len(tracks)), length)
cmin = np.repeat(np.minimum.reduceat(cap, starts), length)
cmax = np.repeat(np.maximum.reduceat(cap, starts), length)
far = np.maximum(cap - cmin, cmax - cap)
img_df = pd.DataFrame({'image': np.arange(len(names)), 'camera': cam_of, 'capture': capture, 'features': per_image,
                       'tracked': np.bincount(F['image'][in_track], minlength=len(names)),
                       'reach_1': np.bincount(T['image'][far >= 1], minlength=len(names)),
                       'reach_5': np.bincount(T['image'][far >= 5], minlength=len(names))})
fig, axes = plt.subplots(len(cams), 1, figsize=(12, 2.2 * len(cams)), sharex=True, sharey=True)
fig.suptitle('Per image: features, those in a track, reaching another capture, reaching 5+ captures away', fontweight='bold')
for ax, c in zip(axes, sorted(cams)):
    d = img_df[img_df.camera == c]
    for col in ['features', 'tracked', 'reach_1', 'reach_5']:
        ax.plot(d.capture, d[col], lw=0.8, label=col)
    ax.set_ylabel(c); ax.grid(alpha=0.3)
axes[0].legend(fontsize=7, ncol=4)
axes[-1].set_xlabel('capture')
show(fig)
print(img_df[['features', 'tracked', 'reach_1', 'reach_5']].describe(percentiles=[.01, .1, .5]).round(0).to_string())

fig, axes = plt.subplots(2, len(cams), figsize=(3.4 * len(cams), 4.2))
for k, c in enumerate(sorted(cams)):
    lens = cams[c].lens
    m = cam_of[F['image']] == c
    for r, (sel, title) in enumerate([(m, 'all features'), (m & in_track, 'in a track')]):
        axes[r, k].hist2d(F['uv'][sel, 0], F['uv'][sel, 1], bins=[48, 27], range=[[0, lens.width], [0, lens.height]], cmap='magma')
        axes[r, k].invert_yaxis(); axes[r, k].set_xticks([]); axes[r, k].set_yticks([])
        axes[r, k].set_title(f'{c}: {title}', fontsize=8)
show(fig)

## Summary

In [ ]:
summary = {
    'run': RUN.name, 'render': run['render'], 'odm_image': run['odm_image'], 'odm_args': run['odm_args'],
    'wrong_px': wrong_px, 'impure_m': impure_m,
    'images': len(names),
    'features': {'total': int(len(F['image'])), 'per_image_median': float(np.median(per_image)),
                 'by_label': {k: int(v) for k, v in by_label.items()}},
    'pairs': {'tried': int(len(pairs)), 'kept': int((pairs.n > 0).sum())},
    'matches': {'total': int(len(M['a'])), 'wrong': int(M['wrong'].sum()),
                'reprojection_px': {q: float(np.percentile(M['err_px'], int(q[1:]))) for q in ['p50', 'p90', 'p99']}},
    'tracks': {'count': int(len(tracks)), 'observations': int(len(T['track'])),
               'length_median': float(tracks.length.median()), 'span_median': float(tracks.span.median()),
               'span_p90': float(tracks.span.quantile(0.9)), 'single_capture': int((tracks.captures == 1).sum()),
               'impure': int(tracks.impure.sum())},
    'per_image_median': {k: float(img_df[k].median()) for k in ['tracked', 'reach_1', 'reach_5']},
}
out = RUN / 'derived'
out.mkdir(exist_ok=True)
(out / 'mapping-sim-features.json').write_text(json.dumps(summary, indent=1))
if debug:
    np.savez_compressed(out / 'mapping-sim-features-matches.npz', **{k: v for k, v in M.items() if k != 'label'})
print(json.dumps(summary, indent=1))